## BigQuery Connection Setup

In [17]:
import warnings
warnings.filterwarnings("ignore")

In [18]:
import os
import pandas as pd
from google.cloud import bigquery

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../config/gcp_credentials.json"

client = bigquery.Client()

query = """
    SELECT * 
    FROM `telco_dataset.live_customers` 
    LIMIT 5
"""

df_live_batch = client.query(query).to_dataframe()

print("Cloud Connection Successful! Here are the first 5 live customers:")
display(df_live_batch.head())

Cloud Connection Successful! Here are the first 5 live customers:


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No_phone_service,...,StreamingTV_No_internet_service,StreamingTV_Yes,StreamingMovies_No_internet_service,StreamingMovies_Yes,Contract_One_year,Contract_Two_year,PaperlessBilling_Yes,PaymentMethod_Credit_card_automatic,PaymentMethod_Electronic_check,PaymentMethod_Mailed_check
0,0,13,18.80,251.25,0,0,1,1,1,0,...,1,0,1,0,0,0,0,0,0,1
1,0,15,18.80,294.95,0,1,1,0,1,0,...,1,0,1,0,1,0,0,0,0,1
2,0,1,18.85,18.85,1,1,0,0,1,0,...,1,0,1,0,0,0,1,0,0,1
3,0,20,18.90,347.65,0,0,0,0,1,0,...,1,0,1,0,0,0,1,1,0,0
4,0,6,19.00,105.50,0,0,1,1,1,0,...,1,0,1,0,1,0,0,1,0,0


## Synthetic Generator

In [26]:
import numpy as np
def trigger_synthetic_generator(client,df_reference,table_id,num_rows=50):
    print(f"\n⚠️ CRITICAL: Live database empty. Generating {num_rows} synthetic customers...")
    fake_data = pd.DataFrame()

    for col in df_reference.columns:
       
        if col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
            mean = df_reference[col].mean()
            std = df_reference[col].std()
            fake_data[col] = np.random.normal(loc=mean, scale=std, size=num_rows)
            fake_data[col] = np.clip(fake_data[col], 0, None) 

        else:
            probs = df_reference[col].value_counts(normalize=True)
            fake_data[col] = np.random.choice(probs.index, p=probs.values, size=num_rows)

    print("Pushing synthetic data to BigQuery via Python API...")
    job = client.load_table_from_dataframe(fake_data, table_id)
    job.result()
    print("✅ Synthetic data successfully injected into the Cloud!")
    return fake_data
print("Synthetic Generator Function is Ready!")


Synthetic Generator Function is Ready!


## The Batch Processing Pipeline

In [27]:
import time
import pickle
import pandas as pd

# loading the trained models
with open('../models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('../models/lin_reg_model.pkl', 'rb') as f:
    lin_reg = pickle.load(f)
with open('../models/log_reg_model.pkl', 'rb') as f:
    log_reg = pickle.load(f)


batch_size = 50
current_offset = 0
max_records_to_test = 500

while current_offset<max_records_to_test:
    print(f"\n[Cloud Sync] Fetching records {current_offset + 1} to {current_offset + batch_size}...")

    query = f"""
        SELECT * 
        FROM `telco_dataset.live_customers` 
        LIMIT {batch_size} OFFSET {current_offset}
    """
    df_batch = client.query(query).to_dataframe()
    
    if df_batch.empty:
        df_historical = pd.read_csv('../data/historical_train_data.csv')
        target_table = "telco_dataset.live_customers" 
        trigger_synthetic_generator(client, df_historical, target_table)
        continue

    
    x_live = df_batch.drop(columns = ['Churn','TotalCharges'],errors = 'coerce')

    #scaling data and predict
    x_live_scaled = scaler.transform(x_live)
    churn_predictions = log_reg.predict(x_live_scaled)
    clv_predictions = lin_reg.predict(x_live_scaled)

    #adding predictions to the df
    df_batch['Predicted_Churn_Risk'] = churn_predictions
    df_batch['Predicted_CLV'] = clv_predictions

    # the smart trigger
    high_value_at_risk = df_batch[(df_batch['Predicted_Churn_Risk'] == 1) & (df_batch['Predicted_CLV'] > 2000)]
    print(f"[Batch {current_offset + 1} to {current_offset + batch_size}] - Fetched {len(df_batch)} customers.")
    print(f"🚨 ALARM: Found {len(high_value_at_risk)} HIGH-VALUE customers at Churn Risk! Sending to Retention Team.\n")


    #update offset
    current_offset += batch_size

    time.sleep(2)
print("\nPipeline Fetching Module Completed Successfully!")
  


[Cloud Sync] Fetching records 1 to 50...
[Batch 1 to 50] - Fetched 50 customers.
🚨 ALARM: Found 0 HIGH-VALUE customers at Churn Risk! Sending to Retention Team.


[Cloud Sync] Fetching records 51 to 100...
[Batch 51 to 100] - Fetched 50 customers.
🚨 ALARM: Found 0 HIGH-VALUE customers at Churn Risk! Sending to Retention Team.


[Cloud Sync] Fetching records 101 to 150...
[Batch 101 to 150] - Fetched 50 customers.
🚨 ALARM: Found 0 HIGH-VALUE customers at Churn Risk! Sending to Retention Team.


[Cloud Sync] Fetching records 151 to 200...
[Batch 151 to 200] - Fetched 50 customers.
🚨 ALARM: Found 0 HIGH-VALUE customers at Churn Risk! Sending to Retention Team.


[Cloud Sync] Fetching records 201 to 250...
[Batch 201 to 250] - Fetched 50 customers.
🚨 ALARM: Found 0 HIGH-VALUE customers at Churn Risk! Sending to Retention Team.


[Cloud Sync] Fetching records 251 to 300...
[Batch 251 to 300] - Fetched 50 customers.
🚨 ALARM: Found 0 HIGH-VALUE customers at Churn Risk! Sending to Retention T